# Documentación del Dataset: Simulación de Consumo Energético

## 1. Descripción General

Este pipeline de generación de datos crea un conjunto de datos sintético y realista diseñado específicamente para tareas de Análisis Exploratorio de Datos (EDA) y entrenamiento de modelos de Machine Learning (Clasificación Multiclase).

El script simula el comportamiento energético de miles de viviendas basándose en características estructurales, demográficas, de equipamiento y factores climáticos. Además, incluye un módulo de inyección de imperfecciones controladas para habilitar la práctica exhaustiva de Data Cleaning y manejo de valores atípicos (Outliers).

## 2. Especificaciones Técnicas

- **Volumen Base:** 10,000 registros únicos.
- **Volumen Final:** 10,100 registros (incluye un 1% de duplicados estructurales intencionales).
- **Variable Objetivo (Target):** `categoria` (Eficiencia Energética: Eficiente, Moderado, Ineficiente).
- **Reproducibilidad:** Semilla aleatoria (`SEED`) fijada en `42` para garantizar resultados consistentes en cada ejecución.
- **Archivo de Salida:** `dataset_crudo.csv`.

## 3. Lógica de Simulación Estadística

Para garantizar el realismo de los datos y evitar distribuciones uniformes planas, se aplicaron las siguientes reglas matemáticas:

- **Distribuciones Log-normales:** Utilizadas para los `metros_cuadrados` y el `ingreso_mensual`. Esto simula la realidad económica, donde la mayoría se concentra en valores medios-bajos, con una "cola larga" de propiedades/ingresos de gran magnitud.

- **Distribución de Poisson:** Empleada para variables de conteo discreto (ej. `habitaciones`, `aires_acondicionados`, `cantidad_personas`), asegurando que no existan valores negativos y respetando proporciones lógicas (ej. 1 habitación por cada ~35m²).

- **Asociación de Variables:** El ecosistema de datos está fuertemente correlacionado. Por ejemplo, una casa antigua tiene mayor probabilidad de tener un mal aislamiento térmico, y los ingresos mensuales condicionan la cantidad de electrodomésticos y el uso de secadoras o losa radiante.

## 4. Diccionario de Datos

A continuación, se detallan las 33 variables generadas, agrupadas por su módulo de origen:

### Características de la Vivienda (`crear_viviendas`)

- **`id` (Entero):** Identificador único del registro (reescrito tras duplicar para dificultar su detección).
- **`tipo_vivienda` (Categórica):** Clasificación del inmueble (`'Casa'`, `'Departamento'`, `'Pequeño Comercio'`).
- **`metros_cuadrados` (Flotante):** Superficie del inmueble acotada entre 30m² y 450m² (antes de outliers).
- **`habitaciones` (Entero):** Cantidad de habitaciones (condicionado por los metros cuadrados).
- **`baños` (Entero):** Cantidad de baños (condicionado por las habitaciones).
- **`antiguedad_vivienda` (Entero):** Años de antigüedad del inmueble (Distribución triangular, moda = 15).
- **`aislamiento` (Categórica):** Calidad del aislamiento térmico (`'Poor'`, `'Average'`, `'Good'`, `'Excellent'`).
- **`eficiencia_construccion` (Categórica):** Calificación constructiva (`'E'`, `'D'`, `'C'`, `'B'`, `'A'`).
- **`paneles_solares` (Booleano):** Indica si la propiedad posee generación solar.

### Demografía y Ocupación (`crear_habitantes`)

- **`cantidad_personas` (Entero):** Habitantes de la propiedad (relacionado con el número de habitaciones).
- **`trabajo_remoto` (Booleano):** Indica si al menos un habitante hace Home Office.
- **`horas_en_casa` (Flotante):** Promedio de horas diarias de ocupación de la vivienda.
- **`ingreso_mensual` (Flotante):** Ingreso estimado del hogar en USD (correlacionado con el tamaño del inmueble).

### Equipamiento y Climatización (`crear_equipamiento`)

- **`aires_acondicionados` (Entero):** Cantidad de equipos de aire acondicionado (0 a 5).
- **`heladeras` (Entero):** Conteo de heladeras/refrigeradores.
- **`televisores` (Entero):** Conteo de televisores en la vivienda.
- **`computadoras` (Entero):** Conteo de computadoras personales o de escritorio.
- **`lavadoras` (Entero):** Conteo de máquinas lavadoras.
- **`secadoras` (Entero):** Conteo de máquinas secadoras de ropa.
- **`cantidad_equipos` (Entero):** Suma total de dispositivos eléctricos, incluyendo pequeños electrodomésticos.
- **`calefaccion` (Booleano):** Existencia de sistema térmico de invierno.
- **`tipo_calefaccion` (Categórica):** Tipo de sistema térmico de invierno (`'Ninguna'`, `'Gas'`, `'Losa Radiante'`, `'Eléctrica'`).
- **`tipo_iluminacion` (Categórica):** Tecnología de iluminación predominante (`'LED'`, `'Mixta'`, `'Incandescente'`).
- **`electrodomesticos_eficientes` (Flotante):** Porcentaje del parque de electrodomésticos considerado de alta eficiencia (%).

### Consumo Energético (`crear_consumo`)

- **`factor_estacional` (Categórica):** Estación del año en la que se registra la medición (`'Verano'`, `'Invierno'`, etc.).
- **`temperatura_media` (Flotante):** Temperatura climática asignada según la estación (con ruido estocástico).
- **`consumo_kwh` (Flotante):** Variable principal de análisis: Consumo total simulado en kilovatios-hora.
- **`uso_horario_pico` (Booleano):** Indica si el mayor consumo se da en franjas de alta demanda (corregida la varianza).
- **`horas_alto_consumo` (Entero):** Franja horaria (0-23) de mayor demanda eléctrica de la propiedad.
- **`tarifa_kwh` (Flotante):** Precio variable del kWh en USD.
- **`costo_estimado` (Flotante):** Facturación estimada (`consumo_kwh` multiplicado por `tarifa_kwh`).

### Puntuación y Variable Objetivo (`crear_score_y_target`)

- **`energy_efficiency_score` (Flotante):** Puntuación continua (Score MVP) calculada exclusivamente con las variables que el modelo verá en producción (Relación entre `consumo_kwh`, `cantidad_equipos` y `horas_alto_consumo`), más un ruido aleatorio para evitar mapeos perfectos.

- **`categoria` (Categórica):** Variable Target para clasificación (`'Eficiente'`, `'Moderado'`, `'Ineficiente'`). Dividida mediante cortes percentiles exactos (33% y 66%) sobre el `energy_efficiency_score` para asegurar que no exista solapamiento de clases y facilitar el aprendizaje de los algoritmos.

## 5. Diseño de Imperfecciones Inyectadas (Data Cleaning)

Para enriquecer el proceso de Ingeniería de Datos y el EDA, el módulo `inyectar_imperfecciones` aplica ruido intencional al dataset base:

- **Valores Nulos (NaN):** Inyectados de forma aleatoria (MCAR - Missing Completely At Random) entre un 2% y 6% estrictamente en las columnas: `ingreso_mensual`, `antiguedad_vivienda`, `electrodomesticos_eficientes` y `horas_en_casa`.

- **Outliers Extremos (Mansiones):** Simulación de propiedades inusualmente grandes, modificando el 0.5% de los datos con superficies infladas entre 500m² y 1,200m².

- **Outliers Extremos (Anomalías Energéticas):** Simulación de consumos absurdos (ej. Minería Cripto clandestina o fallos de lectura), multiplicando aleatoriamente el consumo de algunas viviendas por factores de 4 a 8 (Afecta al 0.5%).

- **Inconsistencias Lógicas:** Creación de registros contradictorios (0.2%), como propiedades con `aislamiento = 'Excellent'` y `paneles_solares = True`, pero con un consumo inexplicablemente altísimo (2,000 - 3,500 kWh), requiriendo investigación y tratamiento.

- **Registros Duplicados:** 100 filas exactas (1%) fueron copiadas de manera idéntica al final del proceso y se les reasignó un nuevo `id` para forzar su identificación analizando las características repetidas del inmueble.

In [10]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN GLOBAL
# ==========================================
SEED = 42
N_REGISTROS = 10000

np.random.seed(SEED)
random.seed(SEED)

# ==========================================
# MÓDULOS DE GENERACIÓN DE DATOS
# ==========================================

def crear_viviendas(n: int) -> pd.DataFrame:
    """
    Genera las características estructurales de las viviendas.
    Utiliza distribuciones log-normales para áreas (evitando colas largas irreales)
    y poisson para conteo de habitaciones.
    """
    df = pd.DataFrame({'id': range(1, n + 1)})

    # 1. Tipo de vivienda (Distribución categórica)
    df['tipo_vivienda'] = np.random.choice(
        ['Casa', 'Departamento', 'Pequeño Comercio'],
        size=n, p=[0.55, 0.40, 0.05]
    )

    # 2. Metros Cuadrados (Log-normal acotada por tipo)
    def generar_m2(tipo):
        if tipo == 'Casa':
            return np.random.lognormal(mean=np.log(120), sigma=0.4)
        elif tipo == 'Departamento':
            return np.random.lognormal(mean=np.log(65), sigma=0.3)
        else:
            return np.random.lognormal(mean=np.log(90), sigma=0.5)

    df['metros_cuadrados'] = df['tipo_vivienda'].apply(generar_m2)
    df['metros_cuadrados'] = np.clip(df['metros_cuadrados'], 30, 450).round(1)

    # 3. Habitaciones y Baños (Poisson correlacionado al tamaño)
    # 1 habitación por cada ~35m2 + base
    lambda_hab = df['metros_cuadrados'] / 35
    df['habitaciones'] = np.random.poisson(lam=lambda_hab)
    df['habitaciones'] = np.clip(df['habitaciones'], 1, 8).astype(int)

    # 1 baño por cada 2-3 habitaciones
    df['baños'] = np.random.poisson(lam=(df['habitaciones'] / 2 + 0.5))
    df['baños'] = np.clip(df['baños'], 1, 5).astype(int)

    # 4. Antigüedad (Distribución Triangular, la mayoría tiene ~15 años)
    df['antiguedad_vivienda'] = np.random.triangular(left=0, mode=15, right=80, size=n).astype(int)

    # 5. Aislamiento y Eficiencia (Correlacionado inversamente con antigüedad)
    # Viviendas más nuevas tienden a tener mejor aislamiento
    prob_base = np.clip(1 - (df['antiguedad_vivienda'] / 80), 0.1, 0.9)

    aislamiento_choices = ['Poor', 'Average', 'Good', 'Excellent']
    df['aislamiento'] = [
        np.random.choice(aislamiento_choices, p=[1-p, (1-p)*0.5, p*0.6, p*0.4] / np.sum([1-p, (1-p)*0.5, p*0.6, p*0.4]))
        for p in prob_base
    ]

    eficiencia_choices = ['E', 'D', 'C', 'B', 'A']
    df['eficiencia_construccion'] = [
        np.random.choice(eficiencia_choices, p=[1-p, (1-p)*0.5, 0.2, p*0.5, p*0.5] / np.sum([1-p, (1-p)*0.5, 0.2, p*0.5, p*0.5]))
        for p in prob_base
    ]

    # 6. Paneles Solares (Binomial: Mayor probabilidad en casas nuevas y grandes)
    prob_solar = np.where(df['tipo_vivienda'] == 'Casa', 0.08, 0.01)
    prob_solar += np.where(df['antiguedad_vivienda'] < 10, 0.05, 0.0)
    df['paneles_solares'] = np.random.binomial(1, np.clip(prob_solar, 0.0, 0.2)).astype(bool)

    return df

def crear_habitantes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera información demográfica y ocupacional fuertemente atada
    a las características del inmueble.
    """
    n = len(df)

    # 1. Cantidad de personas (Poisson condicionado por habitaciones)
    lambda_personas = np.where(df['tipo_vivienda'] == 'Pequeño Comercio',
                               df['habitaciones'] * 1.5,
                               df['habitaciones'] * 0.8 + 0.5)
    df['cantidad_personas'] = np.random.poisson(lam=lambda_personas)
    df['cantidad_personas'] = np.clip(df['cantidad_personas'], 1, 8).astype(int)

    # 2. Trabajo Remoto (Binomial)
    # Comercitos no tienen "trabajo remoto" per se en este contexto
    prob_remoto = np.where(df['tipo_vivienda'] == 'Pequeño Comercio', 0.0, 0.35)
    df['trabajo_remoto'] = np.random.binomial(1, prob_remoto).astype(bool)

    # 3. Horas en casa (Distribución normal ajustada por remoto/tipo)
    base_horas = np.random.normal(loc=12, scale=3, size=n)
    horas = np.where(df['trabajo_remoto'], base_horas + 8, base_horas)
    df['horas_en_casa'] = np.clip(horas, 8, 24).round(1)

    # 4. Ingreso Mensual (Distribución Log-normal, correlacionado con m2 y personas)
    # Simulamos valores en dólares (USD) para estandarización.
    base_ingreso = np.random.lognormal(mean=np.log(1500), sigma=0.6, size=n)
    multiplicador_m2 = df['metros_cuadrados'] / df['metros_cuadrados'].mean()
    df['ingreso_mensual'] = (base_ingreso * multiplicador_m2).round(2)

    return df

def crear_equipamiento(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera el parque de electrodomésticos basándose en poder adquisitivo,
    tamaño del inmueble y cantidad de habitantes.
    """
    n = len(df)

    # Aires acondicionados (Correlación con m2 e ingresos)
    prob_ac = (df['metros_cuadrados'] / 100) * (df['ingreso_mensual'] / 2000)
    df['aires_acondicionados'] = np.random.poisson(lam=np.clip(prob_ac, 0.5, 3))
    df['aires_acondicionados'] = np.clip(df['aires_acondicionados'], 0, 5).astype(int)

    # Heladeras (Al menos 1, más si hay mucha gente/comercios)
    df['heladeras'] = 1 + np.random.binomial(1, p=np.clip(df['cantidad_personas']/10, 0, 0.5))
    df['heladeras'] = np.where(df['tipo_vivienda'] == 'Pequeño Comercio', df['heladeras'] + 1, df['heladeras'])

    # Computadoras y TVs
    df['televisores'] = np.random.poisson(lam=df['cantidad_personas'] * 0.7)
    df['computadoras'] = np.random.poisson(lam=df['cantidad_personas'] * 0.8)
    df['computadoras'] = np.where(df['trabajo_remoto'], df['computadoras'] + 1, df['computadoras'])

    # Lavado
    df['lavadoras'] = np.where(df['tipo_vivienda'] != 'Pequeño Comercio', 1, 0)
    df['secadoras'] = np.random.binomial(1, p=np.clip(df['ingreso_mensual']/10000, 0.05, 0.4))

    df['cantidad_equipos'] = (df['aires_acondicionados'] + df['heladeras'] +
                              df['televisores'] + df['computadoras'] +
                              df['lavadoras'] + df['secadoras'] +
                              np.random.poisson(lam=5, size=n)) # Pequeños electrodomésticos

    # Calefacción y climatización
    df['calefaccion'] = np.random.choice([True, False], size=n, p=[0.8, 0.2])

    def asignar_tipo_calefaccion(row):
        if not row['calefaccion']: return 'Ninguna'
        if row['antiguedad_vivienda'] > 30: return 'Gas'
        if row['ingreso_mensual'] > 4000: return 'Losa Radiante'
        return 'Eléctrica'

    df['tipo_calefaccion'] = df.apply(asignar_tipo_calefaccion, axis=1)

    # Iluminación
    df['tipo_iluminacion'] = np.where(
        df['antiguedad_vivienda'] < 10,
        np.random.choice(['LED', 'Mixta'], size=n, p=[0.8, 0.2]),
        np.random.choice(['LED', 'Mixta', 'Incandescente'], size=n, p=[0.3, 0.5, 0.2])
    )

    # Eficiencia de electrodomésticos (%)
    # Normal distribution, negatively correlated with home age.
    base_eff = 80 - (df['antiguedad_vivienda'] * 0.5) + (df['ingreso_mensual'] / 1000)
    df['electrodomesticos_eficientes'] = np.random.normal(loc=base_eff, scale=10)
    df['electrodomesticos_eficientes'] = np.clip(df['electrodomesticos_eficientes'], 10, 100).round(1)

    return df

def crear_consumo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el consumo eléctrico basándose en las variables anteriores,
    aplicando ruido estocástico (Gaussian noise) para realismo.
    """
    n = len(df)

    # Estacionalidad
    estaciones = ['Verano', 'Invierno', 'Primavera', 'Otoño']
    df['factor_estacional'] = np.random.choice(estaciones, size=n)

    # Temperatura Media (Dependiente de la estación + ruido)
    temp_dict = {'Verano': 28, 'Invierno': 8, 'Primavera': 20, 'Otoño': 15}
    df['temperatura_media'] = df['factor_estacional'].map(temp_dict) + np.random.normal(0, 3, size=n)
    df['temperatura_media'] = df['temperatura_media'].round(1)

    # Consumo Base Físico
    consumo_base = (df['metros_cuadrados'] * 0.8) + (df['cantidad_personas'] * 45)

    # Consumo de Climatización (Penalizado por mal aislamiento)
    factor_aislamiento = df['aislamiento'].map({'Poor': 1.4, 'Average': 1.1, 'Good': 0.8, 'Excellent': 0.6})

    consumo_frio = np.where((df['factor_estacional'] == 'Verano') & (df['aires_acondicionados'] > 0),
                            df['aires_acondicionados'] * 120 * factor_aislamiento, 0)

    consumo_calor = np.where((df['factor_estacional'] == 'Invierno') & (df['tipo_calefaccion'] == 'Eléctrica'),
                             180 * factor_aislamiento, 0)

    # Consumo Equipamiento
    factor_iluminacion = df['tipo_iluminacion'].map({'LED': 0.6, 'Mixta': 1.0, 'Incandescente': 1.8})
    consumo_equipos = (df['cantidad_equipos'] * 25) * factor_iluminacion * (1 - (df['electrodomesticos_eficientes'] * 0.003))

    # Suma de consumos + Impacto de horas en casa
    consumo_total = (consumo_base + consumo_frio + consumo_calor + consumo_equipos) * (df['horas_en_casa'] / 16)

    # Descuento por paneles solares (Reduce entre 20% y 60% del consumo de red)
    descuento_solar = np.where(df['paneles_solares'], np.random.uniform(0.2, 0.6, size=n), 0)
    consumo_total = consumo_total * (1 - descuento_solar)

    # Ruido Aleatorio (Añade imperfección humana)
    ruido = np.random.normal(1.0, 0.15, size=n)
    df['consumo_kwh'] = np.clip(consumo_total * ruido, 50, None).round(2)

    # ==========================================
    # CORRECCIÓN 1: BUG DE VARIANZA RESUELTO
    # Se agregó el parámetro size=n a las funciones binomiales
    # ==========================================
    df['uso_horario_pico'] = np.where(df['trabajo_remoto'],
                                      np.random.binomial(1, 0.7, size=n),
                                      np.random.binomial(1, 0.4, size=n)).astype(bool)

    df['horas_alto_consumo'] = np.where(df['uso_horario_pico'],
                                        np.random.normal(19, 2, size=n),
                                        np.random.normal(14, 4, size=n))
    df['horas_alto_consumo'] = np.clip(df['horas_alto_consumo'], 0, 23).astype(int)

    # Variables Económicas
    df['tarifa_kwh'] = np.random.normal(0.18, 0.02, size=n).round(3) # Precio en USD/kWh con variaciones
    df['costo_estimado'] = (df['consumo_kwh'] * df['tarifa_kwh']).round(2)

    return df

def crear_score_y_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula la categoría de eficiencia basándose EXCLUSIVAMENTE en las variables
    que verá la API en producción (MVP). Esto garantiza que el modelo
    de ML pueda separar las clases sin superposición (overlap).
    """
    n = len(df)

    # ==========================================
    # CORRECCIÓN 2: LÓGICA DE OVERLAP RESUELTA
    # ==========================================

    # 1. Evitar división por cero si cantidad_equipos es 0
    equipos_seguros = np.where(df['cantidad_equipos'] == 0, 1, df['cantidad_equipos'])

    # 2. Score basado puramente en las variables del MVP
    score_mvp = (df['consumo_kwh'] / equipos_seguros) * (1 + (df['horas_alto_consumo'] / 24))

    # 3. Añadir un ruido Gaussiano leve para que los modelos ML deban encontrar patrones
    score_mvp = score_mvp + np.random.normal(0, 3, size=n)

    # Guardar score crudo para análisis en el EDA
    df['energy_efficiency_score'] = score_mvp.round(2)

    # 4. Definir cortes perfectos en tercios (33% y 66%)
    corte_33 = np.percentile(score_mvp, 33)
    corte_66 = np.percentile(score_mvp, 66)

    # 5. Asignar clases sin superposición
    def clasificar(val):
        if val <= corte_33: return 'Eficiente'
        elif val <= corte_66: return 'Moderado'
        else: return 'Ineficiente'

    df['categoria'] = score_mvp.apply(clasificar)

    return df

def inyectar_imperfecciones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inyecta ruido sucio, missing values (NaN), outliers e inconsistencias
    necesarias para la fase de Data Cleaning y EDA.
    """
    df_dirty = df.copy()
    n = len(df_dirty)

    # 1. Valores Faltantes (MCAR - Missing Completely At Random)
    cols_na = ['ingreso_mensual', 'antiguedad_vivienda', 'electrodomesticos_eficientes', 'horas_en_casa']
    for col in cols_na:
        prop_na = np.random.uniform(0.02, 0.06)
        mask = np.random.choice([True, False], size=n, p=[prop_na, 1 - prop_na])
        df_dirty.loc[mask, col] = np.nan

    # 2. Outliers (Casos extremos pero plausibles)
    # Mansiones enormes
    idx_out_m2 = np.random.choice(df_dirty.index, size=int(n * 0.005), replace=False)
    df_dirty.loc[idx_out_m2, 'metros_cuadrados'] = np.random.uniform(500, 1200, size=len(idx_out_m2))

    # Granjas de servidores / Minería cripto escondida (Consumo absurdo)
    idx_out_cons = np.random.choice(df_dirty.index, size=int(n * 0.005), replace=False)
    df_dirty.loc[idx_out_cons, 'consumo_kwh'] = df_dirty.loc[idx_out_cons, 'consumo_kwh'] * np.random.uniform(4, 8)

    # 3. Inconsistencias Lógicas Controladas
    # Casas 'Excellent' pero que consumen excesivamente
    idx_inc = np.random.choice(df_dirty[(df_dirty['aislamiento'] == 'Excellent') &
                                        (df_dirty['paneles_solares'] == True)].index,
                               size=int(n * 0.002), replace=False)
    df_dirty.loc[idx_inc, 'consumo_kwh'] = np.random.uniform(2000, 3500, size=len(idx_inc))

    # 4. Duplicados
    num_duplicados = int(n * 0.01)
    df_duplicados = df_dirty.sample(n=num_duplicados, random_state=SEED)
    df_dirty = pd.concat([df_dirty, df_duplicados]).reset_index(drop=True)

    # Mezclamos el dataset (Shuffle)
    df_dirty = df_dirty.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df_dirty['id'] = range(1, len(df_dirty) + 1) # Rehacer IDs para no delatar los duplicados fácilmente

    return df_dirty

def main():
    print(f" Iniciando generación de dataset de Consumo Energético ({N_REGISTROS} registros)...")

    # Ejecutar pipeline
    df = crear_viviendas(N_REGISTROS)
    df = crear_habitantes(df)
    df = crear_equipamiento(df)
    df = crear_consumo(df)
    df = crear_score_y_target(df)
    df_final = inyectar_imperfecciones(df)

    # ==========================================
    # CORRECCIÓN 3: EXPORTACIÓN ÚNICA DEL CRUDO
    # ==========================================
    filename = 'dataset_crudo.csv'
    df_final.to_csv(filename, index=False, encoding='utf-8')

    # ==========================================
    # REPORTING Y ESTADÍSTICAS
    # ==========================================
    print(f"\n Dataset generado y guardado como: '{filename}'")
    print("-" * 50)
    print(" RESUMEN DE LOS DATOS GENERADOS")
    print("-" * 50)
    print(f"🔹 Número total de registros (con duplicados): {len(df_final)}")
    print(f"🔹 Número de columnas: {df_final.shape[1]}")
    print(f"🔹 Duplicados intencionales agregados: {len(df_final) - N_REGISTROS}")
    print("\n Valores faltantes inyectados por columna:")
    missing = df_final.isnull().sum()
    print(missing[missing > 0].to_string())

    print("\n Distribución de la Variable Objetivo (Categoría corregida en tercios):")
    print(df_final['categoria'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

    print("\n Distribución de Horario Pico (Varianza corregida):")
    print(df_final['uso_horario_pico'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

    print("\n Estadísticas Descriptivas (Muestra reducida):")
    cols_desc = ['metros_cuadrados', 'cantidad_personas', 'consumo_kwh', 'costo_estimado']
    print(df_final[cols_desc].describe().round(2))

    print("\n Matriz de Correlación (Top variables numéricas):")
    cols_corr = ['metros_cuadrados', 'habitaciones', 'aires_acondicionados', 'consumo_kwh', 'energy_efficiency_score']
    print(df_final[cols_corr].corr().round(3))

    print("-" * 50)
    print("¡Listo! Tienes un dataset con distribuciones reales, correlaciones latentes, ruido estadístico y data sucia lista para tu EDA.")

if __name__ == '__main__':
    main()

 Iniciando generación de dataset de Consumo Energético (10000 registros)...

 Dataset generado y guardado como: 'dataset_crudo.csv'
--------------------------------------------------
 RESUMEN DE LOS DATOS GENERADOS
--------------------------------------------------
🔹 Número total de registros (con duplicados): 10100
🔹 Número de columnas: 33
🔹 Duplicados intencionales agregados: 100

 Valores faltantes inyectados por columna:
antiguedad_vivienda             499
horas_en_casa                   362
ingreso_mensual                 415
electrodomesticos_eficientes    445

 Distribución de la Variable Objetivo (Categoría corregida en tercios):
categoria
Ineficiente    34.04 %
Moderado       32.99 %
Eficiente      32.97 %
Name: proportion, dtype: object

 Distribución de Horario Pico (Varianza corregida):
uso_horario_pico
False    50.16 %
True     49.84 %
Name: proportion, dtype: object

 Estadísticas Descriptivas (Muestra reducida):
       metros_cuadrados  cantidad_personas  consumo_kwh  co

In [14]:
import pandas as pd
import numpy as np

def auditar_dataset(filepath='dataset_crudo.csv'):
    print(" Iniciando Auditoría de Calidad de Datos...\n")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(" Error: No se encontró el archivo. Asegúrate de ejecutar el generador primero.")
        return

    # 1. Verificación de Dimensiones
    print(" VERIFICACIÓN DE DIMENSIONES")
    esperado_filas = 10100
    if len(df) == esperado_filas and df.shape[1] == 33:
        print(f"  Correcto: {len(df)} filas y {df.shape[1]} columnas.")
    else:
        print(f"   Error: Se esperaban {esperado_filas} filas y 33 columnas. Hay {len(df)}x{df.shape[1]}")

    # 2. Verificación de Duplicados (excluyendo ID)
    print("\n VERIFICACIÓN DE DUPLICADOS")
    # Como el script anterior reescribió el 'id', buscamos duplicados en el resto de los datos
    duplicados = df.drop(columns=['id']).duplicated().sum()
    if duplicados == 100:
        print(f"   Correcto: Exactamente 100 duplicados intencionales encontrados.")
    else:
        print(f"   Advertencia: Se encontraron {duplicados} duplicados, se esperaban 100.")

    # 3. Verificación de Valores Nulos Controlados
    print("\n VERIFICACIÓN DE DATOS FALTANTES")
    cols_con_nulos = df.columns[df.isnull().any()].tolist()
    nulos_esperados = ['ingreso_mensual', 'antiguedad_vivienda', 'electrodomesticos_eficientes', 'horas_en_casa']

    # Comprobar que SOLO esas columnas tengan nulos
    nulos_inesperados = [c for c in cols_con_nulos if c not in nulos_esperados]
    if not nulos_inesperados:
        print("   Correcto: Los valores nulos están estrictamente en las columnas configuradas.")
    else:
        print(f"   Error: Nulos inesperados en: {nulos_inesperados}")

    # 4. Verificación de Límites Lógicos
    print("\n VERIFICACIÓN DE LÍMITES LÓGICOS")
    errores_limites = 0
    if df['metros_cuadrados'].min() < 30:
        print("   Error: Hay viviendas con menos de 30 m2.")
        errores_limites += 1
    if not df['horas_alto_consumo'].between(0, 23).all():
        print("   Error: 'horas_alto_consumo' fuera del rango 0-23.")
        errores_limites += 1
    if df['consumo_kwh'].min() <= 0:
        print("   Error: Hay consumos eléctricos negativos o cero.")
        errores_limites += 1

    if errores_limites == 0:
        print("   Correcto: Todas las variables numéricas respetan sus límites lógicos.")

    # 5. Verificación de Separación de Clases (Overlap del Target)
    print("\n VERIFICACIÓN DE SOLAPAMIENTO DE CLASES (TARGET)")
    # Agrupamos el score crudo por categoría para ver si los máximos y mínimos se pisan
    stats_clases = df.groupby('categoria')['energy_efficiency_score'].agg(['max', 'min'])

    max_eficiente = stats_clases.loc['Eficiente', 'max']
    min_moderado = stats_clases.loc['Moderado', 'min']
    max_moderado = stats_clases.loc['Moderado', 'max']
    min_ineficiente = stats_clases.loc['Ineficiente', 'min']

    solapamiento = False
    if max_eficiente > min_moderado:
        print(f"   Error: Solapamiento entre Eficiente (máx: {max_eficiente}) y Moderado (mín: {min_moderado})")
        solapamiento = True
    if max_moderado > min_ineficiente:
        print(f"   Error: Solapamiento entre Moderado (máx: {max_moderado}) y Ineficiente (mín: {min_ineficiente})")
        solapamiento = True

    if not solapamiento:
        print("   Correcto: Las fronteras de decisión son perfectas. No hay solapamiento entre categorías.")
        print(f"     ➔ Eficiente termina en: {max_eficiente}")
        print(f"     ➔ Moderado va de: {min_moderado} a {max_moderado}")
        print(f"     ➔ Ineficiente empieza en: {min_ineficiente}")

    print("\n Auditoría finalizada.")

# Ejecutar la prueba
auditar_dataset()

 Iniciando Auditoría de Calidad de Datos...

 VERIFICACIÓN DE DIMENSIONES
  Correcto: 10100 filas y 33 columnas.

 VERIFICACIÓN DE DUPLICADOS
   Correcto: Exactamente 100 duplicados intencionales encontrados.

 VERIFICACIÓN DE DATOS FALTANTES
   Correcto: Los valores nulos están estrictamente en las columnas configuradas.

 VERIFICACIÓN DE LÍMITES LÓGICOS
   Correcto: Todas las variables numéricas respetan sus límites lógicos.

 VERIFICACIÓN DE SOLAPAMIENTO DE CLASES (TARGET)
   Correcto: Las fronteras de decisión son perfectas. No hay solapamiento entre categorías.
     ➔ Eficiente termina en: 42.94
     ➔ Moderado va de: 42.96 a 66.52
     ➔ Ineficiente empieza en: 66.52

 Auditoría finalizada.


In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def ejecutar_pipeline_etl(filepath='dataset_crudo.csv'):
    print(" Iniciando Pipeline ETL...\n")

    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(" Error: No se encontró el dataset crudo.")
        return None

    # ==========================================
    # 1. LIMPIEZA DE DUPLICADOS
    # ==========================================
    # Ignoramos la columna 'id' porque fue reescrita en la simulación
    columnas_sin_id = df.columns.difference(['id'])
    inicial_filas = len(df)
    df = df.drop_duplicates(subset=columnas_sin_id, keep='first')
    print(f" Duplicados eliminados: {inicial_filas - len(df)}")

    # ==========================================
    # 2. IMPUTACIÓN DE VALORES NULOS (MCAR)
    # ==========================================
    # Antigüedad: Imputamos con la mediana global (es robusta a outliers)
    df['antiguedad_vivienda'] = df['antiguedad_vivienda'].fillna(df['antiguedad_vivienda'].median())

    # Ingreso mensual: Mediana agrupada por el tipo de vivienda (más preciso)
    df['ingreso_mensual'] = df['ingreso_mensual'].fillna(
        df.groupby('tipo_vivienda')['ingreso_mensual'].transform('median')
    )

    # Horas en casa: Promedio agrupado por si hace trabajo remoto o no
    df['horas_en_casa'] = df['horas_en_casa'].fillna(
        df.groupby('trabajo_remoto')['horas_en_casa'].transform('mean')
    )

    # Eficiencia de electrodomésticos: Promedio global
    df['electrodomesticos_eficientes'] = df['electrodomesticos_eficientes'].fillna(
        df['electrodomesticos_eficientes'].mean()
    )
    print(" Valores nulos imputados con lógica de negocio.")

    # ==========================================
    # 3. TRATAMIENTO DE OUTLIERS E INCONSISTENCIAS
    # ==========================================
    filas_antes_outliers = len(df)

    # A. Mansiones (Superficie > 450m2 inyectada)
    df = df[df['metros_cuadrados'] <= 450]

    # B. Inconsistencias Lógicas
    # (Aislamiento excelente + paneles, pero consumo desproporcionado > 2000)
    inconsistencias = (df['aislamiento'] == 'Excellent') & (df['paneles_solares'] == True) & (df['consumo_kwh'] > 2000)
    df = df[~inconsistencias]

    # C. Minería Cripto / Anomalías de Consumo (Rango Intercuartílico - IQR)
    # Usamos un multiplicador de 3 para eliminar solo los extremos verdaderamente absurdos
    Q1 = df['consumo_kwh'].quantile(0.25)
    Q3 = df['consumo_kwh'].quantile(0.75)
    IQR = Q3 - Q1
    limite_superior = Q3 + 3 * IQR
    df = df[df['consumo_kwh'] <= limite_superior]

    print(f" Outliers e inconsistencias eliminados: {filas_antes_outliers - len(df)}")

    # ==========================================
    # 4. PREPARACIÓN PARA MACHINE LEARNING
    # ==========================================
    # A. Eliminar variables inútiles o que causan TARGET LEAKAGE
    # energy_efficiency_score se elimina porque es la respuesta matemática del target
    columnas_a_eliminar = ['id', 'energy_efficiency_score', 'costo_estimado']
    df_ml = df.drop(columns=columnas_a_eliminar)

    # B. Codificar Variable Objetivo (Target) a valores numéricos
    target_mapping = {'Eficiente': 0, 'Moderado': 1, 'Ineficiente': 2}
    df_ml['categoria'] = df_ml['categoria'].map(target_mapping)

    # C. Codificación One-Hot para Variables Categóricas
    cols_categoricas = ['tipo_vivienda', 'aislamiento', 'eficiencia_construccion',
                        'tipo_calefaccion', 'tipo_iluminacion', 'factor_estacional']
    df_ml = pd.get_dummies(df_ml, columns=cols_categoricas, drop_first=True)

    # Convertir booleanos a enteros (0 y 1)
    cols_booleanas = df_ml.select_dtypes(include=['bool']).columns
    df_ml[cols_booleanas] = df_ml[cols_booleanas].astype(int)

    # D. Normalización (Escalado Estándar)
    # Vital para modelos basados en distancias (KNN, SVM) o Redes Neuronales
    cols_numericas = df_ml.select_dtypes(include=['float64', 'int64']).columns.drop('categoria')
    scaler = StandardScaler()
    df_ml[cols_numericas] = scaler.fit_transform(df_ml[cols_numericas])

    # ==========================================
    # 5. EXPORTACIÓN
    # ==========================================
    archivo_salida = 'dataset_ml_ready.csv'
    df_ml.to_csv(archivo_salida, index=False)

    print("-" * 50)
    print(f" ETL Completado Exitosamente.")
    print(f" Filas finales para entrenamiento: {len(df_ml)}")
    print(f" Columnas finales (Features): {df_ml.shape[1]}")
    print(f" Guardado como: '{archivo_salida}'")
    print("-" * 50)

    return df_ml

# Ejecutar el proceso
df_modelo = ejecutar_pipeline_etl()

 Iniciando Pipeline ETL...

 Duplicados eliminados: 100
 Valores nulos imputados con lógica de negocio.
 Outliers e inconsistencias eliminados: 158
--------------------------------------------------
 ETL Completado Exitosamente.
 Filas finales para entrenamiento: 9842
 Columnas finales (Features): 41
 Guardado como: 'dataset_ml_ready.csv'
--------------------------------------------------


In [7]:
import pandas as pd

def verificar_dataset_ml(filepath='dataset_ml_ready.csv'):
    print(f" Inspeccionando Dataset para Machine Learning: '{filepath}'\n")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(" Error: No se encontró el dataset. Asegúrate de ejecutar el ETL primero.")
        return

    print(" DIMENSIONES DEL DATASET")
    print(f"  🔹 Filas (Muestras): {df.shape[0]}")
    print(f"  🔹 Columnas (Features + Target): {df.shape[1]}")

    print("\n VERIFICACIÓN DE VALORES NULOS")
    nulos_totales = df.isnull().sum().sum()
    if nulos_totales == 0:
        print("   Perfecto: 0 valores nulos. Los algoritmos no fallarán.")
    else:
        print(f"   Cuidado: Se encontraron {nulos_totales} valores nulos en todo el dataset.")

    print("\n TIPOS DE DATOS (DTYPE)")
    tipos = df.dtypes.unique()
    if all(tipo in ['int64', 'float64', 'int32'] for tipo in tipos):
        print("   Perfecto: Todo el dataset es numérico. Adiós a los errores de texto.")
    else:
        print(f"   Advertencia: Se encontraron tipos no numéricos: {tipos}")

    print("\n DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ('categoria')")
    distribucion = df['categoria'].value_counts(normalize=True).sort_index() * 100
    nombres_clases = {0: 'Eficiente', 1: 'Moderado', 2: 'Ineficiente'}
    for clase, pct in distribucion.items():
        print(f"  🔹 Clase {clase} ({nombres_clases[clase]}): {pct:.2f}%")

    print("\n VISTA PREVIA (Comprobando Estandarización y One-Hot Encoding)")
    # Seleccionamos algunas columnas variadas para ver cómo quedaron transformadas
    columnas_muestra = ['metros_cuadrados', 'consumo_kwh', 'categoria']
    columnas_ohe = [col for col in df.columns if 'tipo_vivienda' in col or 'aislamiento' in col][:3]
    print(df[columnas_muestra + columnas_ohe].head().round(3))

    print("\n Conclusión: El dataset está sanitizado y listo para ser inyectado en un modelo.")

# Ejecutar la verificación
verificar_dataset_ml()

 Inspeccionando Dataset para Machine Learning: 'dataset_ml_ready.csv'

 DIMENSIONES DEL DATASET
  🔹 Filas (Muestras): 9842
  🔹 Columnas (Features + Target): 41

 VERIFICACIÓN DE VALORES NULOS
   Perfecto: 0 valores nulos. Los algoritmos no fallarán.

 TIPOS DE DATOS (DTYPE)
   Perfecto: Todo el dataset es numérico. Adiós a los errores de texto.

 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ('categoria')
  🔹 Clase 0 (Eficiente): 33.10%
  🔹 Clase 1 (Moderado): 33.16%
  🔹 Clase 2 (Ineficiente): 33.73%

 VISTA PREVIA (Comprobando Estandarización y One-Hot Encoding)
   metros_cuadrados  consumo_kwh  categoria  tipo_vivienda_Departamento  \
0            -0.274       -1.060          1                       1.242   
1            -0.308        0.366          2                       1.242   
2             1.371        0.057          0                      -0.805   
3             0.637       -1.283          0                      -0.805   
4            -0.883       -0.340          1                     